# 02 Active Stack And Tuning Policy

This notebook explains the active DAM stack without running the models.

Methodological intent:
- models do **not** all enter at the same feature stage
- that asymmetry is deliberate and fair
- `LEAR` and `XGBoost` are the fair `FS1` models because they rely on explicit regressors
- `Prophet` is delayed until `FS2`, where calendar and holiday structure becomes part of the design
- the first real shortlist happens only after `FS2`


In [1]:
from pathlib import Path
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


In [2]:
display(feature_stage_policy_frame())
display(shortlisting_policy_frame())
display(model_status_frame())
display(tuning_cadence_frame())


,fs_level,feature_scope,active_model_families,prophet_policy,shortlisting_policy,tuning_policy,implementation_status,notes
0,FS0,"Seasonal naive benchmark only: previous-day, p...",naive,Not applicable at FS0.,No shortlisting.,No tuning.,Active,All seasonal naive baselines are evaluated so ...
1,FS1,Explicit endogenous feature foundation only: c...,"lear, xgboost",Prophet is deliberately excluded at FS1.,No shortlisting. LEAR and XGBoost both continu...,"Fast or coarse validation-only tuning, then fr...",Active,FS1 is the explicit endogenous feature foundat...
2,FS2,FS1 endogenous foundation plus forecast-known ...,"lear, xgboost, prophet",Prophet enters for the first time at FS2 becau...,First real shortlist point across the FS2 stack.,"First serious validation-only tuning round, th...",Active,LEAR and XGBoost keep using explicit regressor...
3,FS3,FS2 foundation plus causal exogenous feature f...,shortlisted_fs2_survivors,Carry Prophet into FS3 only if it survives FS2...,Only survivors shortlisted after FS2 continue ...,Mandatory retuning because the feature space c...,Planned scaffold,"Examples include load forecast, wind forecast,..."
4,FS4,Advanced engineered feature layer on top of th...,finalists,Use Prophet at FS4 only with a strong reason a...,Only finalist models continue.,Selective rigorous retuning for surviving fina...,Planned,Main focus is likely XGBoost and possibly LEAR...


,decision_point,policy
0,After FS1,No shortlist. Keep LEAR and XGBoost alive into...
1,After FS2,"First fair shortlist across LEAR, XGBoost, and..."
2,After FS3,Feature-family promotion within the already-sh...
3,After FS4,Finalist selection only.


,model_family,status,first_fs_level,summary
0,naive,Active,FS0,"Seasonal naive previous-day, previous-week, an..."
1,lear,Active,FS1,LEAR is active from FS1 onward as the main lin...
2,xgboost,Active,FS1,XGBoost is active from FS1 onward and is expec...
3,prophet,Active,FS2,Prophet joins only from FS2 onward when calend...
4,arima,Retired,None,ARIMA is removed from the active DAM methodolo...
5,sarima,Retired,None,SARIMA is removed from the active DAM methodol...


,fs_level,search_intensity,validation_role,daily_refit_rule,notes
0,FS0,None,Validation is used only to choose the official...,No structural tuning per origin.,Seasonal naive baselines are deterministic ben...
1,FS1,Fast / coarse,Tune structural hyperparameters on validation ...,Daily walk-forward re-fits only. Do not repeat...,FS1 is not a shortlisting stage.
2,FS2,First serious tuning round,Run the first full validation-only benchmark t...,Daily walk-forward re-fits only. No origin-lev...,FS2 is the first fair shortlist point.
3,FS3,Mandatory retuning,Retune because causal exogenous families mater...,"After the FS3 choice is frozen, daily walk-for...",Apply the same rule again for each materially ...
4,FS4,Selective rigorous retuning,Focus deeper search only on surviving finalists.,Freeze the FS4 design before daily walk-forwar...,Most likely applies to XGBoost and possibly LEAR.


## Tuning snippets

The code below shows the stored tuning placeholders and starter search regions. These are planning aids only; the actual searches stay disabled until the feature stages are executed later.


In [ ]:
display(tuning_snippet_frame())
